### Amplio Pre-processing

Steps:
1. Clean columns [Excel]
2. Split event col into Block and BushRound col [Excel]
3. Add 1/0 decision col and change time to seconds [Py]
4. Drop 'block' from col and add commas to BushRound col [Py]
5. Separate BushRound col into Bush and Round cols [Excel]
6. Drop 'bush' and 'round' text from cols [Py]
7. Sort data: ID --> Block --> Bush --> Round [Excel]
8. Remove (23) individuals who contain no "1" decisions [Py]
9. Remove final bushes of all users [Py]
10. Remove individuals with less than 25 decisions [Py]

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

# Path setup - all paths relative to project root
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path().absolute().parent
AMPLIO_DIR = PROJECT_ROOT / "amplio"

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
''' Step 3: convert Decisions to 0/1 and TotalTime to seconds.'''

# Load data:
amp  = pd.read_csv(AMPLIO_DIR / 'AMPLIO_Mar622_2.csv')
decs = amp['Decision'].tolist()

# Code 0/1 decisions:
decs = [0 if element == 'explore' else element for element in decs]
decs = [1 if element == 'exploit' else element for element in decs]
amp['Decision'] = decs

# Convert TotalTime as seconds
ms_time = amp['TotalTime_ms'].tolist()
ms_time = [element/1000 for element in ms_time]
amp['TotalTime_sec'] = ms_time

amp = amp.drop(['TotalTime_ms'], axis=1)
amp.to_csv(AMPLIO_DIR / 'AMPLIO_Mar622_3.csv')

In [ ]:
''' Step 4: separate event data into block, bush, and round cols.'''

# Load user list (241 unique) data:
amp = pd.read_csv(AMPLIO_DIR / 'AMPLIO_Mar622_3.csv')

# Remove 'block' and keep number:
event = amp['Event'].tolist()
event = [element[5:] for element in event] # drop first 5 elements of each row
amp['Event'] = event # save back to df

# Add commas to bushround col:
bush = amp['Bush'].tolist()
bush_clean = []

# Add a comma after the first 5 elements:
for element in bush:
    if element[5] != 'r':
        element = element[:6] + ',' + element[6:]
        bush_clean.append(element)
    else:
        element = element[:5] + ',' + element[5:]
        bush_clean.append(element)

# Add to dataframe amp and save:
amp['Bush'] = bush_clean
amp.to_csv(AMPLIO_DIR / 'AMPLIO_Mar622_4.csv')

In [ ]:
''' Step 6: further clean bush and round data.'''
amp = pd.read_csv(AMPLIO_DIR / 'AMPLIO_Mar622_5.csv')
bush = amp['Bush'].tolist()
rounds = amp['Round'].tolist()

bush = [element[4:] for element in bush]
amp['Bush'] = bush
rounds = [element[5:] for element in rounds]
amp['Round'] = rounds

amp.to_csv(AMPLIO_DIR / 'AMPLIO_Mar622_6.csv')

In [ ]:
'''
Step 8: Remove individual if no 1 decisions.
'''
amp = pd.read_csv(AMPLIO_DIR / 'AMPLIO_Mar622_7.csv')
users = sorted(set(amp['ID'].tolist()))

print(f'There are {len(users)} users.')
users_with_no_exploits = []
for i in range(len(users)):
    user = users[i]
    user = amp[amp['ID'] == user]
    userblock1 = user[user['Block'] == 1]
    userblock2 = user[user['Block'] == 2]
    userblock1decision = userblock1['Decision'].tolist()
    userblock2decision = userblock2['Decision'].tolist()
    if userblock1decision.count(1) == 0 or userblock2decision.count(1) == 0:
        users_with_no_exploits.append(users[i])

print(f'There are {len(users_with_no_exploits)} users that made no exploit decisions.')

# Remove users with no exploits:
clean1 = amp[~amp['ID'].isin(users_with_no_exploits)]
clean1users = sorted(set(clean1['ID'].tolist()))
print(f'There are {len(clean1users)} users after removing users with no exploits.')
clean1.to_csv(AMPLIO_DIR / 'AMPLIO_Mar622_8.csv')

In [ ]:
'''
Step 9: Remove final bush of all individuals.
'''
amp = pd.read_csv(AMPLIO_DIR / 'AMPLIO_Mar622_8.csv')
users = sorted(set(amp['ID'].tolist()))
print(f'There are {len(users)} users.')

# Remove final bush of all users:
dfs_to_concat = []

for i in range(len(users)):
    user = users[i]
    df = amp[amp['ID'] == user]
    
    df1 = df[df['Block']==1]
    bush1 = df1['Bush'].tolist()
    bush1 = bush1[-1]
    df1 = df1[df1['Bush'] != bush1]

    df2 = df[df['Block']==2]
    bush2 = df2['Bush'].tolist()
    bush2 = bush2[-1]
    df2 = df2[df2['Bush'] != bush2]

    # Save to list for concatenation:
    dfs_to_concat.extend([df1, df2])

amp2 = pd.concat(dfs_to_concat, ignore_index=True)
amp2.to_csv(AMPLIO_DIR / 'AMPLIO_Mar622_9.csv')

In [ ]:
'''
Step 10: Remove individuals with less than 25 decisions.
'''
amp = pd.read_csv(AMPLIO_DIR / 'AMPLIO_Mar622_9.csv')
users = sorted(set(amp['ID'].tolist()))

print(f'There are {len(users)} users.')
users_withlessthan25 = []
for i in range(len(users)):
    user = users[i]
    user = amp[amp['ID'] == user]
    userdecisions = user['Decision'].tolist()
    if len(userdecisions) < 25:
        users_withlessthan25.append(users[i])

print(f'There are {len(users_withlessthan25)} users with less than 25 decisions.')

# Remove users with less than 25 decisions:
clean1 = amp[~amp['ID'].isin(users_withlessthan25)]
clean1users = sorted(set(clean1['ID'].tolist()))
print(f'There are {len(clean1users)} users after removing users with less than 25 decisions.')
clean1.to_csv(AMPLIO_DIR / 'AMPLIO_Mar622_10.csv')